## Imports

In [541]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

from neuro_fuzzy_toolbox import h_ANFIS, Hybrid_learning_algorithm, SONFIS, EarlyStopping, get_measures, Gaussian_MF

In [542]:
import numpy as np

In [543]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [544]:
from ucimlrepo import fetch_ucirepo

# Parkinsons

## Data

In [545]:
parkinsons = fetch_ucirepo(id=174)

X = parkinsons.data.features
y = parkinsons.data.targets

In [546]:
X

,MDVP:Fo,MDVP:Fhi,MDVP:Flo,MDVP:Jitter,MDVP:Jitter,MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer,...,MDVP:APQ,Shimmer:DDA,NHR,HNR,RPDE,DFA,spread1,spread2,D2,PPE
0,119.992,157.302,74.997,0.00784,0.00784,0.00370,0.00554,0.01109,0.04374,0.04374,...,0.02971,0.06545,0.02211,21.033,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,122.400,148.650,113.819,0.00968,0.00968,0.00465,0.00696,0.01394,0.06134,0.06134,...,0.04368,0.09403,0.01929,19.085,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,116.682,131.111,111.555,0.01050,0.01050,0.00544,0.00781,0.01633,0.05233,0.05233,...,0.03590,0.08270,0.01309,20.651,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,116.676,137.871,111.366,0.00997,0.00997,0.00502,0.00698,0.01505,0.05492,0.05492,...,0.03772,0.08771,0.01353,20.644,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,116.014,141.781,110.655,0.01284,0.01284,0.00655,0.00908,0.01966,0.06425,0.06425,...,0.04465,0.10470,0.01767,19.649,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,174.188,230.978,94.261,0.00459,0.00459,0.00263,0.00259,0.00790,0.04087,0.04087,...,0.02745,0.07008,0.02764,19.517,0.448439,0.657899,-6.538586,0.121952,2.657476,0.133050
191,209.516,253.017,89.488,0.00564,0.00564,0.00331,0.00292,0.00994,0.02751,0.02751,...,0.01879,0.04812,0.01810,19.147,0.431674,0.683244,-6.195325,0.129303,2.784312,0.168895
192,174.688,240.005,74.287,0.01360,0.01360,0.00624,0.00564,0.01873,0.02308,0.02308,...,0.01667,0.03804,0.10715,17.883,0.407567,0.655683,-6.787197,0.158453,2.679772,0.131728
193,198.764,396.961,74.904,0.00740,0.00740,0.00370,0.00390,0.01109,0.02296,0.02296,...,0.01588,0.03794,0.07223,19.020,0.451221,0.643956,-6.744577,0.207454,2.138608,0.123306


In [547]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [548]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[ 33 103]


In [549]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[15 44]


In [550]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [551]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [552]:
loader = data.DataLoader(
    data.TensorDataset(x_train, y_train), 
    batch_size = 8, 
    shuffle = True)
x_train = loader.dataset.tensors[0]
y_train = loader.dataset.tensors[1]

## Model & Training

### ANFIS

In [553]:
model = h_ANFIS(
    input_size = 22,
    num_mfs = 1,
    outputs = 1,
    rule_reduced = True,
    output_type='binary',
    dtype=torch.float64
)

model.init_premises(x_train)

### Hybrid Learning Algorithm

In [554]:
loss_fn = nn.functional.binary_cross_entropy
epochs = 100
#optimizer = torch.optim.Adam
#params = {'lr': 0.001}
optimizer = torch.optim.AdamW
params = {'lr': 0.01, 'weight_decay': 0.01}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = EarlyStopping(
    patience=20
)

In [555]:
trainer = Hybrid_learning_algorithm(
    epochs=epochs,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [556]:
Ngrow = 8
dGrow = 3.0
Nsplit = 6
eSplit = 2.6
Nvanish = 4
lVanish = 3

max_iterations = 100

anfis_trainer = trainer

validation = 0.2
sonfis_early_stopping = EarlyStopping(patience=20)
last_training_iteration = False

In [557]:
sonfis = SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [558]:
%time sonfis(model, loader, verbose=True)

Iteration:   0/100 - loss: 1.811173 - validation loss: 1.831052
 -> ANFIS rules: 1

Iteration:   1/100 - loss: 0.433291 - validation loss: 0.425447
 -> ANFIS rules: 2

Iteration:   2/100 - loss: 0.434126 - validation loss: 0.425710
 -> ANFIS rules: 4

Iteration:   3/100 - loss: 0.439366 - validation loss: 0.417673
 -> ANFIS rules: 7

Iteration:   4/100 - loss: 0.415849 - validation loss: 0.398713
 -> ANFIS rules: 9

Iteration:   5/100 - loss: 0.404294 - validation loss: 0.385383
 -> ANFIS rules: 10

Iteration:   6/100 - loss: 0.436223 - validation loss: 0.451006
 -> ANFIS rules: 11

Iteration:   7/100 - loss: 0.397024 - validation loss: 0.423159
 -> ANFIS rules: 16

Iteration:   8/100 - loss: 0.402628 - validation loss: 0.423158
 -> ANFIS rules: 20

Iteration:   9/100 - loss: 0.363074 - validation loss: 0.374989
 -> ANFIS rules: 22

Iteration:  10/100 - loss: 0.355888 - validation loss: 0.373529
 -> ANFIS rules: 25

Iteration:  11/100 - loss: 0.350294 - validation loss: 0.373529
 -> AN

In [559]:
test_measures = get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.847457627118644
Precision: 0.8571428571428571
Recall: 0.9545454545454546
F1: 0.9032258064516129
Confusion Matrix: [[ 8  7]
 [ 2 42]]


In [560]:
train_measures = get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.875
Precision: 0.8771929824561403
Recall: 0.970873786407767
F1: 0.9216589861751152
Confusion Matrix: [[ 19  14]
 [  3 100]]
